# Research Agent RL — Week 3: Multi-Turn GRPO

**Setup:** GPU T4 x1, Internet ON

This notebook trains the LLM with **multi-turn GRPO** — the same core idea
as DeepSeek-R1, but with **real tool execution** at each step:

```
LLM generates action → real HotpotQA search index executes → real result fed back → repeat
```

This closes the main gap vs production agents (like OpenAI Deep Research)
where the LLM sees actual retrieval results during training, not hallucinated ones.

**GRPO mechanics:**
1. For each question, run G=4 independent episodes with real tool calls
2. Score each episode: correctness + format + efficiency
3. Normalise rewards within the group → advantages (no critic network needed)
4. One forward pass over the full trajectory extracts log-probs of LLM-generated tokens only
5. Policy gradient update: push LLM toward higher-reward trajectories

## 1. Check GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU — enable T4 in Settings!')

## 2. Clone repo & install dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/202520030411/Research_Agent_RL.git'
REPO_DIR = '/kaggle/working/Research_Agent_RL'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q transformers peft accelerate datasets pyyaml tqdm bitsandbytes

## 3. Load SFT checkpoint (Week 1 adapter)

We start from the SFT model (already knows the JSON trace format) and
further improve it with GRPO.

In [ ]:
import os, shutil, glob

ADAPTER_DIR = 'checkpoints/qwen-sft/final'

if not os.path.exists(ADAPTER_DIR):
    print('Downloading Week 1 checkpoint...')
    !mkdir -p /tmp/w1_out
    !kaggle kernels output wuyue22/week3multigrpo -p /tmp/w1_out
    candidates = [
        '/tmp/w1_out/qwen-sft-adapter',
        '/tmp/w1_out/Research_Agent_RL/checkpoints/qwen-sft/final',
    ]
    src = None
    for c in candidates:
        if os.path.exists(c) and os.path.exists(os.path.join(c, 'adapter_config.json')):
            src = c
            break
    if src:
        os.makedirs('checkpoints/qwen-sft', exist_ok=True)
        shutil.copytree(src, ADAPTER_DIR)
        print(f'Adapter copied from {src} → {ADAPTER_DIR}')
    else:
        ckpts = sorted(glob.glob('/tmp/w1_out/Research_Agent_RL/checkpoints/qwen-sft/checkpoint-*'))
        if ckpts:
            shutil.copytree(ckpts[-1], ADAPTER_DIR)
            print(f'Using latest checkpoint: {ckpts[-1]}')
        else:
            print('[ERROR] No checkpoint found.')
else:
    print(f'Checkpoint already present at {ADAPTER_DIR}')

!ls -lh {ADAPTER_DIR}


In [ ]:
import os, json, torch
from peft import PeftModel, get_peft_model, LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# Cut memory fragmentation — lets the allocator reuse freed blocks more
# aggressively, which helps when per-episode forward/backward keeps
# returning activations to the pool.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

ADAPTER_DIR = 'checkpoints/qwen-sft/final'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

with open(f'{ADAPTER_DIR}/adapter_config.json') as f:
    base_model_name = json.load(f).get('base_model_name_or_path', 'Qwen/Qwen2.5-0.5B-Instruct')

print(f'Loading base model: {base_model_name}')
# bfloat16 (not float16) — fp16 + LoRA + Adam is numerically unstable
# and produces NaN losses in GRPO. bf16 has the same memory savings with
# the dynamic range of fp32, which is what the GRPO log-prob math needs.
base = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map={'': 0},
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

# is_trainable=True is the critical flag — without it, PeftModel.from_pretrained
# loads the adapter in eval mode with requires_grad=False on every parameter,
# and the Adam optimizer ends up with an empty parameter list.
model = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=True)
model.config.use_cache = False
model.enable_input_require_grads()   # needed for gradient flow through LoRA

# Gradient checkpointing — recomputes attention activations during backward
# instead of storing them. ~2x memory savings on the T4 at the cost of ~25%
# extra compute per backward pass. Essential for multi-turn GRPO on 16GB.
model.gradient_checkpointing_enable()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable:,} / {total:,}  ({100*trainable/total:.3f}%)')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')
assert trainable > 0, 'No trainable parameters — check is_trainable=True on PeftModel.from_pretrained'

## 4. Load HotpotQA & build real search index

In [ ]:
from datasets import load_dataset
from agent.tools import ToolExecutor

N_TRAIN = 200
N_VAL   = 100

print('Loading HotpotQA...')
hf       = load_dataset('hotpot_qa', 'distractor', trust_remote_code=True)
train_hf = hf['train'].shuffle(seed=42)
val_hf   = hf['validation']

train_questions = [
    {'question': train_hf[i]['question'], 'answer': train_hf[i]['answer']}
    for i in range(N_TRAIN)
]
val_questions = [
    {'question': val_hf[i]['question'], 'answer': val_hf[i]['answer']}
    for i in range(N_VAL)
]

# Build a document-level retrieval index from HotpotQA titles + sentences.
tool_executor = ToolExecutor(top_k=2)
tool_executor.build_from_hotpotqa(val_hf, index_path='data/tool_index.jsonl')
print(f'Tool index: {len(tool_executor)} documents')
print(f'Train questions: {len(train_questions)} | Val: {len(val_questions)}')

## 5. Smoke-test: one interactive episode

In [ ]:
from rl.multi_turn_grpo import collect_episode
from data.sft_dataset import SYSTEM_PROMPT

model.eval()
q = val_questions[0]
ep = collect_episode(
    q['question'], q['answer'],
    model, tokenizer, tool_executor,
    system_prompt=SYSTEM_PROMPT,
    max_steps=4, temperature=0.1, device=DEVICE,
)

print(f'Q      : {ep.question}')
print(f'Gold   : {ep.gold_answer}')
print(f'Correct: {ep.correct}  |  Steps: {ep.n_steps}')
print(f'\n--- Full trajectory ---')
print(ep.full_text[-800:])   # show end of trajectory

## 5b. Filter training questions to the learnable zone

For each question, run K=3 rollouts with the SFT model and real tools at
temperature=0.5. Keep questions where n_correct in {0, 1, 2} out of 3 —
excluding only the "all 3 correct" bucket (too easy). Even all-wrong
groups contribute useful gradient from format-reward variation across
rollouts, so we keep them in training.

**Why temperature=0.5 (not the 0.9 default):** at T=0.9 the SFT model
cannot hold JSON format — in the previous run the full histogram was
{0: 1000}, meaning every single rollout failed before emitting a valid
`answer` action. T=0.5 is close enough to the T=0.1 baseline (26% EM
at τ=0.85) to preserve format while still producing enough rollout
variance to get non-degenerate advantages.

Expected runtime: ~20–30 min on T4 for 200 candidate questions.

In [ ]:
from rl.multi_turn_grpo import filter_learnable_questions

# Pre-filter the training candidates. min_correct=0 (not 1) keeps all-wrong
# groups too — they still contribute gradient from format-reward variation
# across rollouts. The only bucket we exclude is "all K correct" (too easy).
#
# temperature=0.5: previous run used the 0.9 default → JSON format broke →
# {0: 1000} histogram. 0.5 preserves format while still giving rollout variance.
#
# max_steps=5: HotpotQA is 2-hop, so the natural trajectory is
# search(entity_1) → read(entity_1) → search(entity_2) → read(entity_2)
# → answer. That's 5 steps. At max_steps=3 the model never gets to emit
# `answer` and every rollout registers as incorrect.
learnable_train = filter_learnable_questions(
    questions=train_questions,
    model=model,
    tokenizer=tokenizer,
    tool_executor=tool_executor,
    system_prompt=SYSTEM_PROMPT,
    K=3,
    max_steps=5,
    min_correct=0,        # keep 0/1/2-correct groups; exclude only 3/3
    max_correct=2,
    temperature=0.5,
    device=DEVICE,
    verbose=True,
)

print(f'\nFiltered train set: {len(learnable_train)} / {len(train_questions)}')
assert len(learnable_train) >= 50, (
    f'Only {len(learnable_train)} learnable questions after filter. '
    'Expected >=50. Something is wrong with the rollouts — inspect the '
    'zone histogram above before starting GRPO training.'
)

## 6. Multi-turn GRPO training

In [ ]:
from rl.multi_turn_grpo import train as mt_grpo_train
import torch

torch.cuda.empty_cache()

# Hyperparameters — tuned for T4 16GB + 12h Kaggle cap.
#
# G=4: four rollouts per question → real advantage baseline.
# (R_i - mean)/std is much tighter than G=2.
#
# temperature=0.5: at the default 0.9 the SFT model lost JSON format
# and every rollout failed (zone {0: 1000}). 0.5 preserves format.
#
# max_steps=5: matches the SFT training trajectory length for 2-hop
# HotpotQA (search → read → search → read → answer). At max_steps=3
# the model never reaches the `answer` action.
#
# batch_size=1 + per-episode backward: peak VRAM = ONE trajectory's
# activations (~1.5k tokens × Qwen-0.5B bf16).
#
# n_epochs=200: each "epoch" is one gradient update on a randomly-
# sampled question (not a full pass over the data).

# Fall back to raw train_questions if the filter cell (5b) was skipped.
if 'learnable_train' not in globals():
    print('WARNING: learnable_train not defined — skipping filter, using raw train_questions')
    print('         expect ~50% tied groups contributing zero gradient')
    learnable_train = train_questions

log = mt_grpo_train(
    model=model,
    tokenizer=tokenizer,
    tool_executor=tool_executor,
    train_questions=learnable_train,
    val_questions=val_questions,
    system_prompt=SYSTEM_PROMPT,
    output_dir='checkpoints/mt-grpo',
    n_epochs=200,
    batch_size=1,
    G=4,
    lr=5e-6,
    max_steps=5,
    alpha=0.1,
    beta=0.05,
    epsilon=0.05,
    temperature=0.5,
    val_every=20,
    save_every=40,
    device=DEVICE,
)

print('Training complete!')

## 7. Evaluate trained model

In [ ]:
from rl.multi_turn_grpo import collect_episode
from tqdm import tqdm

model.eval()
N_EVAL = 100

correct, total_steps = 0, 0
results = []

for q in tqdm(val_questions[:N_EVAL], desc='Evaluating'):
    ep = collect_episode(
        q['question'], q['answer'],
        model, tokenizer, tool_executor,
        system_prompt=SYSTEM_PROMPT,
        max_steps=5, temperature=0.1, device=DEVICE,
    )
    correct     += int(ep.correct)
    total_steps += ep.n_steps
    results.append({
        'question': ep.question, 'gold': ep.gold_answer,
        'correct': ep.correct, 'n_steps': ep.n_steps,
    })

acc       = correct / N_EVAL
avg_steps = total_steps / N_EVAL
print(f'\nMulti-turn GRPO model')
print(f'  Accuracy : {acc:.3f}')
print(f'  Avg steps: {avg_steps:.1f}')
print(f'  Efficiency (acc/steps): {acc/max(avg_steps,1):.4f}')

## 8. Save everything

In [ ]:
import json, shutil

# Copy final adapter to output
shutil.copytree('checkpoints/mt-grpo/final', '/kaggle/working/mt_grpo_adapter', dirs_exist_ok=True)
print('Adapter saved → /kaggle/working/mt_grpo_adapter')

# Eval results
with open('/kaggle/working/mt_grpo_eval_results.json', 'w') as f:
    json.dump({
        'summary': {'accuracy': acc, 'avg_steps': avg_steps,
                    'efficiency': acc / max(avg_steps, 1), 'n_eval': N_EVAL},
        'results': results,
    }, f, indent=2)
print('Eval results saved → /kaggle/working/mt_grpo_eval_results.json')

# Training log
with open('/kaggle/working/mt_grpo_training_log.json', 'w') as f:
    json.dump(log, f, indent=2)
print('Training log saved → /kaggle/working/mt_grpo_training_log.json')

print('\nAll outputs in Kaggle Output tab.')